https://www.consumerfinance.gov/data-research/consumer-complaints/search/?dateRange=All&date_received_max=2026-03-09&date_received_min=2011-12-01&page=1&searchField=all&size=25&sort=created_date_desc&tab=List

**import the required libaries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from pathlib import Path
load_dotenv("env_variables")

DATASET_DIR = Path(os.getenv("DATASET_DIR"))

In [ ]:
df=pd.read_csv(DATASET_DIR / "NLP"/"complaints_processed.csv")

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df['product'].value_counts()

In [ ]:
df.dropna(axis=0,inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('wordnet')


In [ ]:

def clean_text(text):
    # Remove HTML tags
    clean_text = re.sub(r'<.*?>', '', text)
    
    # Remove links
    clean_text = re.sub(r'http\S+', '', clean_text)
    
    # Remove punctuation
    clean_text = clean_text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove emojis
    clean_text = clean_text.encode('ascii', 'ignore').decode('ascii')
    
    # Tokenize text
    tokens = word_tokenize(clean_text)

    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word.lower() not in stop_words]
    
    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    lemmatized_text = [lemmatizer.lemmatize(word) for word in filtered_tokens]
    
    # Join the tokens back into a single string
    clean_text = ' '.join(lemmatized_text)
    
    return clean_text


In [ ]:
df.head()

In [ ]:
df.narrative=df.narrative.apply(lambda x:clean_text(x))

In [ ]:
df.head()

**Data Preprocessing**

In [ ]:
x=df['narrative'].tolist()
y=df['product']

**Feature Enginerring**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()


In [ ]:
X_train,X_test,Y_train,Y_test=train_test_split(x,y,test_size=.2,random_state=42)

In [ ]:
# Fit and transform the text data into TF-IDF features for training set
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform the text data into TF-IDF features for testing set
X_test_tfidf = tfidf.transform(X_test)

**Model Bulding**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
RFC=RandomForestClassifier()

In [ ]:
RFC.fit(X_train_tfidf,Y_train)

In [ ]:
Y_predict=RFC.predict(X_test_tfidf)

**Model Evlaution**

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
accuracy=accuracy_score(Y_test,Y_predict)

In [ ]:
accuracy

In [ ]:
X_test[:10]

In [ ]:
Y_predict[:10]

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, f1_score

# Compute confusion matrix
conf_matrix = confusion_matrix(Y_test, Y_predict)
print("Confusion Matrix:")
print(conf_matrix)

In [ ]:
# Compute precision
precision = precision_score(Y_test, Y_predict, average='weighted')
print("Precision:", precision)

In [ ]:
# Compute F1 score
f1 = f1_score(Y_test, Y_predict, average='weighted')
print("F1 Score:", f1)